<a href="https://colab.research.google.com/github/selim679/Backblaze-panne-disques-durs/blob/main/notebookbacklaze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

salimfekihsalem_backblaze1_path = kagglehub.dataset_download('salimfekihsalem/backblaze1')
salimfekihsalem_backblaze_pkl1_path = kagglehub.dataset_download('salimfekihsalem/backblaze-pkl1')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

import pandas as pd
import numpy as np
import os, gc
from tqdm import tqdm

MODEL    = 'ST4000DM000'
DATA_DIR = '/kaggle/working/data/'
os.makedirs(DATA_DIR, exist_ok=True)

PKL_TRAIN = DATA_DIR + 'Lab1-2017-Full-ST4000DM000.pkl'
PKL_TEST  = DATA_DIR + 'Lab1-2016-Q4-ST4000DM000.pkl'

folders_2017 = [
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q1_2017',
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q2_2017',
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q3_2017',
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q4_2017',
]
folders_2016_q4 = [
    '/kaggle/input/datasets/salimfekihsalem/backblaze1/data_Q4_2016',
]

def load_filtered_and_save(folders, pkl_path, model=MODEL):
    """Charge CSV un par un, filtre sur le modèle, sauvegarde en PKL."""

    # Colonnes à garder (métadonnées + SMART brutes uniquement)
    META = ['date', 'serial_number', 'model', 'capacity_bytes', 'failure']

    chunks = []
    total_rows = 0

    for folder in folders:
        if not os.path.exists(folder):
            print(f"⚠️  Manquant : {folder}")
            continue

        files = sorted([
            os.path.join(folder, f)
            for f in os.listdir(folder) if f.endswith('.csv')
        ])
        print(f"\n📂 {os.path.basename(folder)} → {len(files)} fichiers")

        for file in tqdm(files):
            try:
                # 1. Lire seulement les colonnes nécessaires
                # D'abord récupère les colonnes disponibles
                cols_available = pd.read_csv(file, nrows=0).columns.tolist()

                # Garde META + toutes les colonnes smart_X_raw (pas normalized)
                smart_raw = [c for c in cols_available
                             if 'smart_' in c and '_normalized' not in c]
                cols_to_read = META + smart_raw
                cols_to_read = [c for c in cols_to_read if c in cols_available]

                # 2. Lire avec filtre colonnes
                tmp = pd.read_csv(file, usecols=cols_to_read, low_memory=False)

                # 3. Filtrer sur le modèle
                tmp = tmp[tmp['model'] == model].copy()

                if tmp.empty:
                    continue

                # 4. Réduire la mémoire immédiatement
                for col in tmp.select_dtypes('float64').columns:
                    tmp[col] = tmp[col].astype('float32')
                for col in tmp.select_dtypes('int64').columns:
                    tmp[col] = tmp[col].astype('int32')

                total_rows += len(tmp)
                chunks.append(tmp)
                del tmp

            except Exception as e:
                print(f"❌ {file}: {e}")

        # Libérer après chaque quarter
        gc.collect()

    if not chunks:
        raise ValueError("Aucune donnée !")

    print(f"\n🔗 Concaténation de {len(chunks)} chunks ({total_rows} lignes)...")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    ram = df.memory_usage(deep=True).sum() / 1e9
    print(f"✅ Shape : {df.shape}  |  RAM : {ram:.2f} GB")
    print(f"   Pannes : {df['failure'].sum()}")

    print(f"💾 Sauvegarde → {pkl_path}")
    df.to_pickle(pkl_path)
    print("✅ PKL sauvegardé !")

    return df

# --- Génération TRAIN ---
print("=" * 50)
print("GÉNÉRATION TRAIN (2017)")
print("=" * 50)
df = load_filtered_and_save(folders_2017, PKL_TRAIN)

# --- Génération TEST ---
print("\n" + "=" * 50)
print("GÉNÉRATION TEST (2016 Q4)")
print("=" * 50)
df_t = load_filtered_and_save(folders_2016_q4, PKL_TEST)

print("\n✅ TERMINÉ")
print(f"Fichiers générés :")
print(f"  {PKL_TRAIN}  ({os.path.getsize(PKL_TRAIN)/1e6:.1f} MB)")
print(f"  {PKL_TEST}   ({os.path.getsize(PKL_TEST)/1e6:.1f} MB)")

In [ ]:
# ============================================================
# CHARGEMENT RAPIDE (après upload des PKL)
# ============================================================
import pandas as pd

df   = pd.read_pickle('/kaggle/input/datasets/salimfekihsalem/backblaze-pkl1/Lab1-2017-Full-ST4000DM000.pkl')
df_t = pd.read_pickle('/kaggle/input/datasets/salimfekihsalem/backblaze-pkl1/Lab1-2016-Q4-ST4000DM000.pkl')

print("TRAIN :", df.shape,   "| Pannes :", df['failure'].sum())
print("TEST  :", df_t.shape, "| Pannes :", df_t['failure'].sum())

TRAIN : (12237899, 50) | Pannes : 1058
TEST  : (3196552, 50) | Pannes : 234


In [ ]:
df.head()

,date,serial_number,model,capacity_bytes,failure,smart_1_raw,smart_2_raw,smart_3_raw,smart_4_raw,smart_5_raw,...,smart_225_raw,smart_226_raw,smart_240_raw,smart_241_raw,smart_242_raw,smart_250_raw,smart_251_raw,smart_252_raw,smart_254_raw,smart_255_raw
0,2017-01-01,Z305B2QN,ST4000DM000,-2122489856,0,58173272.0,NaN,0.0,8.0,0.0,...,NaN,NaN,8947.0,3.078054e+10,8.290869e+09,NaN,NaN,NaN,NaN,NaN
1,2017-01-01,Z302A0YH,ST4000DM000,-2122489856,0,75626904.0,NaN,0.0,19.0,0.0,...,NaN,NaN,16788.0,2.164812e+10,1.620139e+11,NaN,NaN,NaN,NaN,NaN
2,2017-01-01,Z305BT0W,ST4000DM000,-2122489856,0,48893128.0,NaN,0.0,7.0,0.0,...,NaN,NaN,7771.0,2.616829e+10,1.589215e+10,NaN,NaN,NaN,NaN,NaN
3,2017-01-01,Z302A0YE,ST4000DM000,-2122489856,0,192617016.0,NaN,0.0,12.0,0.0,...,NaN,NaN,17308.0,2.126106e+10,2.363861e+11,NaN,NaN,NaN,NaN,NaN
4,2017-01-01,Z302PGH8,ST4000DM000,-2122489856,0,83211536.0,NaN,0.0,20.0,0.0,...,NaN,NaN,13560.0,1.619163e+10,7.198671e+10,NaN,NaN,NaN,NaN,NaN


In [ ]:

def analyze_columns(df, keyword='smart'):
    """
    Analyse toutes les colonnes contenant un mot-clé.
    Retourne un DataFrame avec les statistiques de chaque colonne.
    """
    # Sélectionner uniquement les colonnes contenant le mot-clé
    smart_cols = [c for c in df.columns if keyword in c.lower()]

    results = []
    for col in smart_cols:
        n_unique = df[col].nunique(dropna=True)
        col_min  = df[col].min(skipna=True)
        col_max  = df[col].max(skipna=True)
        n_nan    = df[col].isna().sum()

        is_constant = (n_unique <= 1) or (col_min == col_max)

        results.append({
            'colonne'      : col,
            'n_unique'     : n_unique,
            'min'          : col_min,
            'max'          : col_max,
            'n_nan'        : n_nan,
            'est_constante': is_constant
        })

    return pd.DataFrame(results)

# --- Analyse ---
print("Analyse des colonnes SMART...")
col_stats = analyze_columns(df, keyword='smart')

# Listes constantes vs informatives
const_cols = col_stats[col_stats['est_constante']]['colonne'].tolist()
info_cols  = col_stats[~col_stats['est_constante']]['colonne'].tolist()

print(f"\n📊 Colonnes SMART totales    : {len(col_stats)}")
print(f"   Colonnes CONSTANTES       : {len(const_cols)}")
print(f"   Colonnes INFORMATIVES     : {len(info_cols)}")
print(f"\nColonnes constantes : {const_cols}")
print(f"\nColonnes informatives : {info_cols}")

Analyse des colonnes SMART...

📊 Colonnes SMART totales    : 45
   Colonnes CONSTANTES       : 24
   Colonnes INFORMATIVES     : 21

Colonnes constantes : ['smart_2_raw', 'smart_3_raw', 'smart_8_raw', 'smart_10_raw', 'smart_11_raw', 'smart_13_raw', 'smart_15_raw', 'smart_22_raw', 'smart_191_raw', 'smart_195_raw', 'smart_196_raw', 'smart_200_raw', 'smart_201_raw', 'smart_220_raw', 'smart_222_raw', 'smart_223_raw', 'smart_224_raw', 'smart_225_raw', 'smart_226_raw', 'smart_250_raw', 'smart_251_raw', 'smart_252_raw', 'smart_254_raw', 'smart_255_raw']

Colonnes informatives : ['smart_1_raw', 'smart_4_raw', 'smart_5_raw', 'smart_7_raw', 'smart_9_raw', 'smart_12_raw', 'smart_183_raw', 'smart_184_raw', 'smart_187_raw', 'smart_188_raw', 'smart_189_raw', 'smart_190_raw', 'smart_192_raw', 'smart_193_raw', 'smart_194_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw', 'smart_240_raw', 'smart_241_raw', 'smart_242_raw']


In [ ]:
analyze_columns(df, keyword='smart')

,colonne,n_unique,min,max,n_nan,est_constante
0,smart_1_raw,10052895,0.0,2.441406e+08,202,False
1,smart_2_raw,0,NaN,NaN,12237899,True
2,smart_3_raw,1,0.0,0.000000e+00,202,True
3,smart_4_raw,210,1.0,1.506000e+03,202,False
4,smart_5_raw,768,0.0,5.324800e+04,202,False
5,smart_7_raw,12096941,2.0,2.814717e+14,202,False
6,smart_8_raw,0,NaN,NaN,12237899,True
7,smart_9_raw,39782,0.0,3.991900e+04,202,False
8,smart_10_raw,1,0.0,0.000000e+00,202,True
9,smart_11_raw,0,NaN,NaN,12237899,True


In [ ]:
analyze_columns(df, keyword='')

,colonne,n_unique,min,max,n_nan,est_constante
0,date,361,2017-01-01,2017-12-31,0,False
1,serial_number,35189,S3000A9T,Z307Y2X9,0,False
2,model,1,ST4000DM000,ST4000DM000,0,True
3,capacity_bytes,3,-2122489856,-1,0,False
4,failure,2,0,1,0,False
5,smart_1_raw,10052895,0.0,244140608.0,202,False
6,smart_2_raw,0,NaN,NaN,12237899,True
7,smart_3_raw,1,0.0,0.0,202,True
8,smart_4_raw,210,1.0,1506.0,202,False
9,smart_5_raw,768,0.0,53248.0,202,False


In [ ]:

print(f"Dimensions AVANT suppression : {df.shape}")

df   = df.drop(columns=const_cols, errors='ignore')
df_t = df_t.drop(columns=const_cols, errors='ignore')

print(f"Dimensions APRÈS suppression : {df.shape}")
print(f"Colonnes supprimées          : {len(const_cols)}")

# Vérification : aucune variabilité dans les colonnes supprimées
print("\n Vérification des colonnes supprimées (toutes doivent être constantes) :")
print(col_stats[col_stats['est_constante']][['colonne','n_unique','min','max']].to_string(index=False))

Dimensions AVANT suppression : (12237899, 29)
Dimensions APRÈS suppression : (12237899, 26)
Colonnes supprimées          : 24

 Vérification des colonnes supprimées (toutes doivent être constantes) :
      colonne  n_unique  min  max
  smart_2_raw         0  NaN  NaN
  smart_3_raw         1  0.0  0.0
  smart_8_raw         0  NaN  NaN
 smart_10_raw         1  0.0  0.0
 smart_11_raw         0  NaN  NaN
 smart_13_raw         0  NaN  NaN
 smart_15_raw         0  NaN  NaN
 smart_22_raw         0  NaN  NaN
smart_191_raw         1  0.0  0.0
smart_195_raw         0  NaN  NaN
smart_196_raw         0  NaN  NaN
smart_200_raw         0  NaN  NaN
smart_201_raw         0  NaN  NaN
smart_220_raw         0  NaN  NaN
smart_222_raw         0  NaN  NaN
smart_223_raw         0  NaN  NaN
smart_224_raw         0  NaN  NaN
smart_225_raw         0  NaN  NaN
smart_226_raw         0  NaN  NaN
smart_250_raw         0  NaN  NaN
smart_251_raw         0  NaN  NaN
smart_252_raw         0  NaN  NaN
smart_254_raw     

In [ ]:
# ============================================================
# SECTION 2.3 — Nettoyage et sélection finale
# ============================================================

import gc

# ---- ÉTAPE 1 : Supprimer colonnes entièrement vides --------
before = df.shape[1]
df   = df.dropna(axis=1, how='all')
df_t = df_t.dropna(axis=1, how='all')
print(f"Colonnes entièrement vides supprimées : {before - df.shape[1]}")

# ---- ÉTAPE 2 : Supprimer lignes avec valeurs critiques manquantes ----
critical_cols = ['date', 'serial_number', 'failure']
before = len(df)
df   = df.dropna(subset=critical_cols)
df_t = df_t.dropna(subset=critical_cols)
print(f"Lignes supprimées (valeurs critiques manquantes) : {before - len(df)}")

# ---- ÉTAPE 3 : Colonnes avec valeurs manquantes restantes ----
nan_cols = df.columns[df.isna().any()].tolist()
print(f"\nColonnes avec NaN restants ({len(nan_cols)}) :")
print(df[nan_cols].isna().sum().sort_values(ascending=False).head(10))

# ---- ÉTAPE 4 : Remplacement des NaN par 0 ----
# Justification : les attributs SMART manquants signifient
# souvent que le disque ne supporte pas cet attribut → valeur neutre = 0
df[nan_cols]   = df[nan_cols].fillna(0)
df_t[nan_cols] = df_t[nan_cols].fillna(0)
print(f"\nNaN restants après remplacement : {df.isna().sum().sum()}")

# ---- ÉTAPE 5 : Séparation normaux / défaillants ----
df_failed  = df[df['failure'] == 1].copy()
df_normal  = df[df['failure'] == 0].copy()
print(f"\nDisques défaillants (failure=1) : {len(df_failed)} lignes")
print(f"Disques normaux    (failure=0) : {len(df_normal)} lignes")

Colonnes entièrement vides supprimées : 0
Lignes supprimées (valeurs critiques manquantes) : 0

Colonnes avec NaN restants (0) :
Series([], dtype: float64)

NaN restants après remplacement : 0

Disques défaillants (failure=1) : 1058 lignes
Disques normaux    (failure=0) : 12236841 lignes


In [ ]:
# ---- ÉTAPE 6 & 7 : Sélection des variables SMART informatives ----
META_COLS = ['date', 'serial_number', 'failure']

# Variables brutes uniquement (pas normalized, déjà supprimées avant)
smart_raw_cols = [c for c in df.columns
                  if 'smart_' in c and '_normalized' not in c]

# Liste finale des colonnes
final_cols = META_COLS + smart_raw_cols

print(f"Colonnes métadonnées  : {len(META_COLS)}")
print(f"Colonnes SMART brutes : {len(smart_raw_cols)}")
print(f"Total colonnes finales: {len(final_cols)}")
print(f"\nColonnes SMART conservées : {smart_raw_cols}")

Colonnes métadonnées  : 3
Colonnes SMART brutes : 24
Total colonnes finales: 27

Colonnes SMART conservées : ['smart_1_raw', 'smart_3_raw', 'smart_4_raw', 'smart_5_raw', 'smart_7_raw', 'smart_9_raw', 'smart_10_raw', 'smart_12_raw', 'smart_183_raw', 'smart_184_raw', 'smart_187_raw', 'smart_188_raw', 'smart_189_raw', 'smart_190_raw', 'smart_191_raw', 'smart_192_raw', 'smart_193_raw', 'smart_194_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw', 'smart_240_raw', 'smart_241_raw', 'smart_242_raw']


In [ ]:
# ---- ÉTAPE 8 & 9 : Appliquer les mêmes colonnes au test ----
# Garder seulement les colonnes finales
df   = df[[c for c in final_cols if c in df.columns]].copy()
df_t = df_t[[c for c in final_cols if c in df_t.columns]].copy()

# Vérification stricte
assert list(df.columns) == list(df_t.columns), "❌ Colonnes différentes entre train et test !"
print("✅ Colonnes identiques entre train et test")
print(f"   Ordre identique : {list(df.columns) == list(df_t.columns)}")

# ---- ÉTAPE 10 : Vérification finale ----
print(f"\n=== VÉRIFICATION FINALE ===")
print(f"TRAIN : {df.shape}  | NaN : {df.isna().sum().sum()} | Types : {df.dtypes.value_counts().to_dict()}")
print(f"TEST  : {df_t.shape} | NaN : {df_t.isna().sum().sum()} | Types : {df_t.dtypes.value_counts().to_dict()}")
display(df.head(3))

✅ Colonnes identiques entre train et test
   Ordre identique : True

=== VÉRIFICATION FINALE ===
TRAIN : (12237899, 24)  | NaN : 0 | Types : {dtype('float32'): 11, dtype('float64'): 10, dtype('O'): 2, dtype('int32'): 1}
TEST  : (3196552, 24) | NaN : 0 | Types : {dtype('float32'): 11, dtype('float64'): 10, dtype('O'): 2, dtype('int32'): 1}


,date,serial_number,failure,smart_1_raw,smart_4_raw,smart_5_raw,smart_7_raw,smart_9_raw,smart_12_raw,smart_183_raw,...,smart_190_raw,smart_192_raw,smart_193_raw,smart_194_raw,smart_197_raw,smart_198_raw,smart_199_raw,smart_240_raw,smart_241_raw,smart_242_raw
0,2017-01-01,Z305B2QN,0,58173272.0,8.0,0.0,388359773.0,9195.0,8.0,0.0,...,24.0,0.0,33904.0,24.0,0.0,0.0,0.0,8947.0,3.078054e+10,8.290869e+09
1,2017-01-01,Z302A0YH,0,75626904.0,19.0,0.0,785458463.0,17043.0,19.0,0.0,...,19.0,0.0,39656.0,19.0,0.0,0.0,0.0,16788.0,2.164812e+10,1.620139e+11
2,2017-01-01,Z305BT0W,0,48893128.0,7.0,0.0,316494047.0,7857.0,7.0,0.0,...,30.0,0.0,9073.0,30.0,0.0,0.0,0.0,7771.0,2.616829e+10,1.589215e+10


In [ ]:
# ============================================================
# SECTION 2.4 — Sauvegarde en Pickle
# ============================================================

import os, time, gc

DATA_DIR = '/kaggle/working/data/'
os.makedirs(DATA_DIR, exist_ok=True)  # 🔥 FIX

PKL_TRAIN = DATA_DIR + 'Lab1-2017-Full-ST4000DM000.pkl'
PKL_TEST  = DATA_DIR + 'Lab1-2016-Q4-ST4000DM000.pkl'

print("💾 Sauvegarde TRAIN...")
df.to_pickle(PKL_TRAIN)

print("💾 Sauvegarde TEST...")
df_t.to_pickle(PKL_TEST)

# Vérification
df_check = pd.read_pickle(PKL_TRAIN)
print(f"\n✅ PKL rechargé : {df_check.shape} == {df.shape} → {df_check.shape == df.shape}")

# Temps de chargement
t0 = time.time()
pd.read_pickle(PKL_TRAIN)
print(f"⚡ Chargement PKL  : {time.time()-t0:.3f}s")

# Taille
print(f"\nTaille fichiers :")
print(f"  Train PKL : {os.path.getsize(PKL_TRAIN)/1e6:.1f} MB")
print(f"  Test  PKL : {os.path.getsize(PKL_TEST)/1e6:.1f} MB")

del df_check
gc.collect()

💾 Sauvegarde TRAIN...
💾 Sauvegarde TEST...

✅ PKL rechargé : (12237899, 24) == (12237899, 24) → True
⚡ Chargement PKL  : 3.965s

Taille fichiers :
  Train PKL : 1739.0 MB
  Test  PKL : 450.7 MB


0

In [ ]:

import pandas as pd
import numpy as np
import os, gc

DATA_DIR  = '/kaggle/working/data/'
PKL_TRAIN = DATA_DIR + 'Lab1-2017-Full-ST4000DM000.pkl'
PKL_TEST  = DATA_DIR + 'Lab1-2016-Q4-ST4000DM000.pkl'

# ÉTAPE 1 — Chargement
df   = pd.read_pickle(PKL_TRAIN)
df_t = pd.read_pickle(PKL_TEST)
print(f"Train chargé : {df.shape}")
print(f"Test  chargé : {df_t.shape}")

Train chargé : (12237899, 24)
Test  chargé : (3196552, 24)


In [ ]:
# ÉTAPE 2 & 3 — Supprimer colonnes 'normalized'
norm_cols = [c for c in df.columns if 'normalized' in c.lower()]
print(f"Colonnes 'normalized' trouvées : {len(norm_cols)}")
print(norm_cols)

df   = df.drop(columns=norm_cols, errors='ignore')
df_t = df_t.drop(columns=norm_cols, errors='ignore')
print(f"\nAprès suppression normalized → Train : {df.shape} | Test : {df_t.shape}")

Colonnes 'normalized' trouvées : 0
[]

Après suppression normalized → Train : (12237899, 24) | Test : (3196552, 24)


In [ ]:
# ÉTAPE 4 — Supprimer 'model' et 'capacity_bytes'
cols_to_drop = ['model', 'capacity_bytes']
df   = df.drop(columns=cols_to_drop, errors='ignore')
df_t = df_t.drop(columns=cols_to_drop, errors='ignore')
print(f"Après suppression model/capacity → Train : {df.shape} | Test : {df_t.shape}")

Après suppression model/capacity → Train : (12237899, 24) | Test : (3196552, 24)


In [ ]:
# ÉTAPE 5 & 6 — Convertir 'date' en datetime
df['date']   = pd.to_datetime(df['date'])
df_t['date'] = pd.to_datetime(df_t['date'])

# Vérification
print(f"Type colonne 'date' train : {df['date'].dtype}")
print(f"Type colonne 'date' test  : {df_t['date'].dtype}")
print(f"Exemple dates : {df['date'].head(3).tolist()}")

Type colonne 'date' train : datetime64[ns]
Type colonne 'date' test  : datetime64[ns]
Exemple dates : [Timestamp('2017-01-01 00:00:00'), Timestamp('2017-01-01 00:00:00'), Timestamp('2017-01-01 00:00:00')]


In [ ]:
# ÉTAPE 7 & 8 & 9 — Tri chronologique par disque puis par date
df   = df.sort_values(['serial_number', 'date']).reset_index(drop=True)
df_t = df_t.sort_values(['serial_number', 'date']).reset_index(drop=True)

# Vérification ordre chronologique sur un disque exemple
disk_example = df['serial_number'].iloc[0]
dates_disk   = df[df['serial_number'] == disk_example]['date'].tolist()
is_sorted    = all(dates_disk[i] <= dates_disk[i+1] for i in range(len(dates_disk)-1))
print(f"Disque exemple : {disk_example}")
print(f"Dates ordonnées chronologiquement : {is_sorted}")
print(f"Premières dates : {dates_disk[:5]}")

Disque exemple : S3000A9T
Dates ordonnées chronologiquement : True
Premières dates : [Timestamp('2017-06-03 00:00:00'), Timestamp('2017-06-04 00:00:00'), Timestamp('2017-06-05 00:00:00'), Timestamp('2017-06-06 00:00:00'), Timestamp('2017-06-07 00:00:00')]


In [ ]:
# ÉTAPE 10, 11 & 12 — Gestion valeurs manquantes
print(f"NaN avant remplacement — Train : {df.isna().sum().sum()} | Test : {df_t.isna().sum().sum()}")

# Colonnes avec NaN
nan_cols_train = df.columns[df.isna().any()].tolist()
print(f"Colonnes avec NaN : {nan_cols_train}")

# Remplacement par 0
df   = df.fillna(0)
df_t = df_t.fillna(0)

print(f"NaN après remplacement — Train : {df.isna().sum().sum()} | Test : {df_t.isna().sum().sum()}")
print("✅ Aucun NaN restant")

NaN avant remplacement — Train : 0 | Test : 0
Colonnes avec NaN : []
NaN après remplacement — Train : 0 | Test : 0
✅ Aucun NaN restant


In [ ]:
# ÉTAPE 13 — Analyse de l'impact
print("=== ANALYSE FINALE SECTION 3 ===")
print(f"Shape train      : {df.shape}")
print(f"Shape test       : {df_t.shape}")
print(f"Colonnes finales : {df.columns.tolist()}")
print(f"\nPlage de dates train : {df['date'].min()} → {df['date'].max()}")
print(f"Plage de dates test  : {df_t['date'].min()} → {df_t['date'].max()}")
print(f"\nNombre de disques uniques train : {df['serial_number'].nunique()}")
print(f"Nombre de disques uniques test  : {df_t['serial_number'].nunique()}")
display(df.head(5))

=== ANALYSE FINALE SECTION 3 ===
Shape train      : (12237899, 24)
Shape test       : (3196552, 24)
Colonnes finales : ['date', 'serial_number', 'failure', 'smart_1_raw', 'smart_4_raw', 'smart_5_raw', 'smart_7_raw', 'smart_9_raw', 'smart_12_raw', 'smart_183_raw', 'smart_184_raw', 'smart_187_raw', 'smart_188_raw', 'smart_189_raw', 'smart_190_raw', 'smart_192_raw', 'smart_193_raw', 'smart_194_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw', 'smart_240_raw', 'smart_241_raw', 'smart_242_raw']

Plage de dates train : 2017-01-01 00:00:00 → 2017-12-31 00:00:00
Plage de dates test  : 2016-10-01 00:00:00 → 2016-12-31 00:00:00

Nombre de disques uniques train : 35189
Nombre de disques uniques test  : 34970


,date,serial_number,failure,smart_1_raw,smart_4_raw,smart_5_raw,smart_7_raw,smart_9_raw,smart_12_raw,smart_183_raw,...,smart_190_raw,smart_192_raw,smart_193_raw,smart_194_raw,smart_197_raw,smart_198_raw,smart_199_raw,smart_240_raw,smart_241_raw,smart_242_raw
0,2017-06-03,S3000A9T,0,107687208.0,33.0,0.0,33158496.0,728.0,27.0,0.0,...,25.0,2.0,1785.0,25.0,0.0,0.0,0.0,503.0,5.442997e+09,304800640.0
1,2017-06-04,S3000A9T,0,132850464.0,33.0,0.0,33315265.0,752.0,27.0,0.0,...,25.0,2.0,2337.0,25.0,0.0,0.0,0.0,525.0,9.374590e+09,319131616.0
2,2017-06-05,S3000A9T,0,176573120.0,33.0,0.0,33376638.0,776.0,27.0,0.0,...,25.0,0.0,3547.0,25.0,0.0,0.0,0.0,545.0,9.374697e+09,351515712.0
3,2017-06-06,S3000A9T,0,218899664.0,33.0,0.0,33435680.0,800.0,27.0,0.0,...,25.0,0.0,4728.0,25.0,0.0,0.0,0.0,565.0,9.374803e+09,384307744.0
4,2017-06-07,S3000A9T,0,19331576.0,33.0,0.0,33495492.0,824.0,27.0,0.0,...,25.0,0.0,5912.0,25.0,0.0,0.0,0.0,584.0,9.374907e+09,415046976.0
